# Principio de Sustitución de Liskov (LSP) — Cafetería

**Dominio propio:** métodos de pago en la caja de la cafetería.

El LSP dice que una subclase debe poder **sustituir** a su clase base sin romper el programa: debe respetar el contrato (mismos tipos de retorno, mismas precondiciones/postcondiciones).

Muestro una subclase que **sí** cumple el contrato y un contraejemplo que **lo viola** (con el error resultante).

## Clase base y caja registradora

`MetodoPago.cobrar(monto)` define el contrato: devuelve un `dict` con el resultado de la transacción. La `Caja` trabaja contra `MetodoPago` sin conocer la implementación concreta.

In [1]:
class MetodoPago:
    def __init__(self, titular: str) -> None:
        self.titular: str = titular
        self.transacciones: int = 0

    def cobrar(self, monto: float) -> dict:
        # Contrato: devuelve SIEMPRE un dict con estas llaves
        self.transacciones += 1
        return {"aprobado": True, "monto": monto, "metodo": "generico"}

    def resumen(self) -> str:
        return f"{self.titular}: {self.transacciones} transacciones"


class Caja:
    def __init__(self, metodo: MetodoPago) -> None:
        self.metodo: MetodoPago = metodo
        self.recaudado: float = 0.0

    def procesar(self, monto: float) -> None:
        resultado = self.metodo.cobrar(monto)
        # La Caja CONFIA en el contrato: resultado es dict con 'aprobado' y 'monto'
        if resultado["aprobado"]:
            self.recaudado += resultado["monto"]
            print(f"Cobro aprobado por ${resultado['monto']:.2f} via {resultado['metodo']}")

## Subclase que CUMPLE el LSP

`PagoTarjeta` respeta el contrato: recibe un `monto`, devuelve un `dict` con las mismas llaves. Es sustituible por `MetodoPago` en cualquier `Caja`.

In [2]:
class PagoTarjeta(MetodoPago):
    def __init__(self, titular: str, ultimos4: str) -> None:
        super().__init__(titular)
        self.ultimos4: str = ultimos4

    def cobrar(self, monto: float) -> dict:
        self.transacciones += 1
        return {"aprobado": True, "monto": monto, "metodo": f"tarjeta ****{self.ultimos4}"}


# Sustituible: la Caja funciona igual con la clase base o con la subclase
caja_generica = Caja(MetodoPago("Anonimo"))
caja_tarjeta = Caja(PagoTarjeta("Ana", "1234"))

caja_generica.procesar(12000)
caja_tarjeta.procesar(12000)
print("Recaudado tarjeta:", caja_tarjeta.recaudado)

Cobro aprobado por $12000.00 via generico
Cobro aprobado por $12000.00 via tarjeta ****1234
Recaudado tarjeta: 12000.0


## Contraejemplo que VIOLA el LSP

`PagoPuntos` rompe el contrato: en vez de devolver un `dict`, devuelve un `bool`. La `Caja` espera un `dict`, así que al sustituir el método base por este falla en tiempo de ejecución.

In [4]:
class PagoPuntos(MetodoPago):
    def __init__(self, titular: str, puntos: int) -> None:
        super().__init__(titular)
        self.puntos: int = puntos

    def cobrar(self, monto: float) -> bool:  # <-- rompe el contrato: bool en vez de dict
        self.transacciones += 1
        return self.puntos >= monto

In [6]:
# Uso que VIOLA LSP: reventara porque cobrar() no devuelve un dict
caja_puntos = Caja(PagoPuntos("Luis", puntos=50000))
caja_puntos.procesar(12000)  # TypeError: 'bool' no es subscriptable

TypeError: 'bool' object is not subscriptable

### Análisis

`PagoTarjeta` **es sustituible** por `MetodoPago` porque respeta el tipo de retorno (`dict`) y las llaves esperadas.

`PagoPuntos` **viola el LSP**: cambia el tipo de retorno a `bool`, así que `resultado["aprobado"]` lanza `TypeError`. Aunque hereda de `MetodoPago`, no puede reemplazarlo sin romper la `Caja`.

La corrección sería que `PagoPuntos.cobrar` también devuelva un `dict` con las mismas llaves, respetando el contrato del padre.